In [1]:
import torch

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("CUDA is not available.")

import importlib

drive = importlib.import_module("google.colab.drive")

getattr(drive, "mount")("/content/drive")

from pathlib import Path

root_path = Path("/content/drive/MyDrive/indoor_object_detection")

annotations = [
    'annotation/annotation_s1.xml',
    'annotation/annotation_s2.xml',
    'annotation/annotation_s3.xml',
    'annotation/annotation_s4.xml',
    'annotation/annotation_s5.xml',
    'annotation/annotation_s6.xml',
]

sequences = [
    'sequence_1',
    'sequence_2',
    'sequence_3',
    'sequence_4',
    'sequence_5',
    'sequence_6',
]

annotations = [root_path / annotation for annotation in annotations]

sequences = [root_path / sequence for sequence in sequences]


Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
height = 720
width = 1280

split_ratios = {
    "train": 0.8,
    "val": 0.1,
    "test": 0.1
}

In [3]:
from pathlib import Path
import xml.etree.ElementTree as ET


def convert_xml_to_boxes(
    xml_path: str | Path,
    image_height: int,
    image_width: int,
) -> dict[str, list]:
    xml_path = Path(xml_path)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    result: dict[str, list] = {}

    for image_elem in root.findall("./images/image"):
        image_filename = image_elem.attrib["file"]

        boxes = []

        for box_elem in image_elem.findall("box"):
            top = float(box_elem.attrib["top"])
            left = float(box_elem.attrib["left"])
            width = float(box_elem.attrib["width"])
            height = float(box_elem.attrib["height"])

            label_elem = box_elem.find("label")
            if label_elem is None or label_elem.text is None:
                raise ValueError(
                    f"Missing label for box in image {image_filename}"
                )

            label = label_elem.text.strip()

            center_x = left + width / 2.0
            center_y = top + height / 2.0

            center_x /= image_width
            center_y /= image_height
            norm_width = width / image_width
            norm_height = height / image_height

            boxes.append([label, center_x, center_y, norm_width, norm_height])

        result[image_filename] = boxes

    return result

all_boxes = [convert_xml_to_boxes(annotation, height, width) for annotation in annotations]